First step !




In [1]:
import os
from pathlib import Path
import json
import requests
import re
import ipywidgets as widgets
from IPython.display import display
from dotenv import load_dotenv
from openai import OpenAI
from ollama import chat
from anthropic import Anthropic
from IPython.display import Markdown, display


 
 

In [2]:
client = OpenAI()

In [3]:
for _d in [Path.cwd(), *Path.cwd().parents]:
    _env = _d / ".env"
    if _env.is_file():
        load_dotenv(_env, override=True)
        break

API_KEY_WEATHER = os.environ["WEATHERAPI_KEY"]
API_KEY_EXCHANGE = os.environ["EXCHANGE_RATE_API_KEY"]

In [4]:
SYSTEM_PROMPT = """
You are a strict routing assistant.

Your job is to choose exactly ONE tool and extract parameters.

Available tools:
1. getWeather(city)
2. calculateMath(expression)
3. getExchangeRate(currencyCode)
4. generalChat(userInput)

Rules:
- Return ONLY valid JSON (no text before or after)
- Never explain anything
- Never add comments

Routing rules:
- If input is a math expression → calculateMath
- If input asks about weather → getWeather
- If input is currency or exchange → getExchangeRate
- Otherwise → generalChat

Parameter rules:
- city must be in English (e.g., Tel Aviv, Haifa)
- currencyCode must be uppercase (USD, EUR, ILS)
- expression must be raw math string

Return format:
{
  "tool": "tool_name",
  "parameters": {
    "param_name": "value"
  }
}
"""

In [5]:
HISTORY_FILE = "history.json"

# 🔹 טעינת היסטוריה
def load_history():
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, "r", encoding="utf-8") as f:
            history = json.load(f)
        print("ברוך שובך 👋")
        return history
    else:
        return []


# 🔹 שמירת היסטוריה
def save_history(history):
    with open(HISTORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)


# 🔹 איפוס היסטוריה
def reset_history():
    if os.path.exists(HISTORY_FILE):
        os.remove(HISTORY_FILE)
    print("ההיסטוריה אופסה 🧹")
    return []

In [6]:
reset_history()

ההיסטוריה אופסה 🧹


[]

In [7]:
load_dotenv(override=True)
client = OpenAI()  

In [8]:
tools = [
    {
        "name": "getWeather",
        "description": "Get current weather for a city",
        "parameters": {
            "city": "string"
        }
    },
    {
        "name": "calculateMath",
        "description": "Calculate a math expression",
        "parameters": {
            "expression": "string"
        }
    },
    {
        "name": "getExchangeRate",
        "description": "Get exchange rate to ILS",
        "parameters": {
            "currencyCode": "string"
        }
    },
    {
        "name": "generalChat",
        "description": "General conversation"
    }
]

In [9]:
def calculateMath(expression):
    try:
        return eval(expression)
    except Exception:
        return "Invalid expression"

In [10]:
def get_weather_description(code):
    if code == 0:
        return "Clear sky"
    elif code in [1, 2, 3]:
        return "Cloudy"
    elif code in [61, 63, 65]:
        return "Rain"
    elif code in [71, 73, 75]:
        return "Snow"
    else:
        return "Unknown"

In [11]:
 
def getWeather(city: str):
    try:
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}"
        geo_response = requests.get(geo_url).json()

        if "results" not in geo_response:
            return "City not found"

        lat = geo_response["results"][0]["latitude"]
        lon = geo_response["results"][0]["longitude"]

 
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_response = requests.get(weather_url).json()

        temp = weather_response["current_weather"]["temperature"]
        wind = weather_response["current_weather"]["windspeed"]
        weather_code = weather_response["current_weather"]["weathercode"]
        description = get_weather_description(weather_code)

        return f"{city}: {temp}°C, {description}, wind {wind} km/h"

    except Exception as e:
        return f"Error: {str(e)}"

In [12]:
def getExchangeRate(currencyCode: str):
    currencyCode = currencyCode.upper()

    url = f"https://v6.exchangerate-api.com/v6/{API_KEY_EXCHANGE}/latest/{currencyCode}"

    try:
        response = requests.get(url)
        data = response.json()

        if data.get("result") != "success":
            return "Ошибка API"

        rate = data["conversion_rates"].get("ILS")

        if rate is None:
            return "Нет данных по ILS"

        return f"1 {currencyCode} = {rate} ILS"

    except Exception as e:
        return f"Ошибка: {str(e)}"

In [ ]:
def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    return match.group(0) if match else "{}"


def openai_router(user_input):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        temperature=0   
    )

    raw_output = response.choices[0].message.content
    print("DEBUG ROUTER:", raw_output)

    clean_json = extract_json(raw_output)

    try:
        return json.loads(clean_json)
    except:
        return {
            "tool": "generalChat",
            "parameters": {"userInput": user_input}
        }

In [14]:
print(openai_router("what's the weather in Paris tomorrow?"))

DEBUG ROUTER: {
  "tool": "getWeather",
  "parameters": {
    "city": "Paris"
  }
}
{'tool': 'getWeather', 'parameters': {'city': 'Paris'}}


In [15]:
def chat(user_input, history):
    route = openai_router(user_input)
    tool = route.get("tool")
    params = route.get("parameters", {})

    print("DEBUG ROUTE:", route)

    if tool == "getWeather":
        result = getWeather(params.get("city", ""))

    elif tool == "calculateMath":
        result = calculateMath(params.get("expression", ""))

    elif tool == "getExchangeRate":
        result = getExchangeRate(params.get("currencyCode", ""))

    else:
        result = generalChat(history, user_input)

    # save history
    history.append({"role": "user", "content": user_input})
    history.append({"role": "assistant", "content": str(result)})

    save_history(history)

    return result

In [19]:
def generalChat(context, userInput):
    messages = [
        {"role": m["role"], "content": str(m["content"])}
        for m in context
    ] + [
        {"role": "user", "content": str(userInput)}
    ]

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )

    return response.choices[0].message.content

In [26]:
def main():
    history = load_history()

    input_box = widgets.Text(
        placeholder='Type your message...',
        description='You:',
        layout=widgets.Layout(width='70%')
    )

    output_box = widgets.Output()

    def on_submit(change):
        nonlocal history

        user_input = input_box.value   # ✅ вот так

        if not user_input.strip():
            return

        if not user_input.strip():
            return

        if user_input.lower() == "/exit":
            with output_box:
                print("Goodbye 👋")
            return

        if user_input.lower() == "/reset":
            history = reset_history()
            with output_box:
                print("History cleared 🧹")
            input_box.value = ""
            return

        response = chat(user_input, history)

        with output_box:
            print(f"You: {user_input}")
            print(f"Bot: {response}")
            print("")

        input_box.value = ""

    input_box.on_submit(on_submit)

    display(input_box, output_box)

In [27]:
main()



ברוך שובך 👋


C:\Users\Yana Naydenova\AppData\Local\Temp\ipykernel_17540\1795782648.py:44: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  input_box.on_submit(on_submit)


Text(value='', description='You:', layout=Layout(width='70%'), placeholder='Type your message...')

Output()